# Pose Classification Training in Google Colab

This notebook trains a lightweight pose classifier to distinguish **bersedia** (ready) vs **berlari** (running) using MediaPipe for pose extraction and TensorFlow Keras for a tiny dense model. The resulting TFLite model can be integrated into the Flutter app.

## 1️⃣ Setup – Install dependencies

In [ ]:
!pip install -q tensorflow==2.14 mediapipe opencv-python scikit-learn tqdm

## 2️⃣ Prepare training data

Create a folder `data/` with two subfolders: `bersedia/` and `berlari/`.
Upload at least 20‑30 images per class (full‑body poses, clear lighting).
The folder structure must look like:
```
data/
├─ bersedia/
│  ├─ img1.jpg
│  └─ ...
└─ berlari/
   ├─ img1.jpg
   └─ ...
```

In [ ]:
from google.colab import files

print('Upload a zip of your `data/` folder (e.g., data.zip)')
uploaded = files.upload()

import zipfile, os
for fn in uploaded.keys():
    with zipfile.ZipFile(fn, 'r') as zip_ref:
        zip_ref.extractall('.')
    print(f'Extracted {fn}')

## 3️⃣ Training script (same as `train_pose_classifier.py`)

In [ ]:
import os, json, numpy as np, cv2, mediapipe as mp, tensorflow as tf, sklearn
from sklearn.model_selection import train_test_split

DATA_DIR = 'data'
BONE_CONNECTIONS = [
    (11, 13), (13, 15), (12, 14), (14, 16), (11, 12), (11, 23), (12, 24),
    (23, 25), (25, 27), (24, 26), (26, 28), (23, 24)
]

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.5)

def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)
    if not results.pose_landmarks:
        return None
    lm = results.pose_landmarks.landmark
    coords = np.array([(p.x, p.y, p.z) for p in lm], dtype=np.float32)
    # normalize
    left_hip, right_hip = coords[23], coords[24]
    hip_center = (left_hip + right_hip) / 2
    coords -= hip_center
    left_shoulder, right_shoulder = coords[11], coords[12]
    torso_len = np.linalg.norm((left_shoulder + right_shoulder) / 2)
    if torso_len > 1e-6:
        coords /= torso_len
    features = []
    features.extend(coords.flatten())
    for s, e in BONE_CONNECTIONS:
        vec = coords[e] - coords[s]
        features.extend(vec)
    for s, e in BONE_CONNECTIONS:
        vec = coords[e] - coords[s]
        features.append(np.dot(vec, vec))
    return np.array(features, dtype=np.float32)

# Build dataset
X, y = [], []
label_map = {}
for idx, label in enumerate(sorted(os.listdir(DATA_DIR))):
    label_path = os.path.join(DATA_DIR, label)
    if not os.path.isdir(label_path):
        continue
    label_map[label] = idx
    for f in os.listdir(label_path):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            fp = os.path.join(label_path, f)
            feats = extract_features(fp)
            if feats is not None:
                X.append(feats)
                y.append(idx)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(label_map), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=80, batch_size=32, validation_data=(X_test, y_test))

# Export TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()
with open('pose_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

# Save labels
with open('pose_labels.json', 'w') as f:
    json.dump(label_map, f)

print('✅ Training complete. Files: pose_classifier.tflite, pose_labels.json')

## 4️⃣ Download the model files

```python
from google.colab import files
files.download('pose_classifier.tflite')
files.download('pose_labels.json')
```

## 5️⃣ Integrate into Flutter

1. Copy the downloaded `pose_classifier.tflite` and `pose_labels.json` into `assets/models/` in your Flutter project.
2. Add them to `pubspec.yaml` under `assets:`.
3. Run `flutter pub get`.
4. Use the provided `lib/ml/pose_classifier.dart` to run inference.